# Deep Learning for Time Series Forecasting

Using LSTM networks for sequence-to-value prediction.

1. **Windowed Dataset** - Creating sequences for supervised learning
2. **LSTM Forecaster** - Architecture and training
3. **Multi-step Forecasting** - Predicting multiple future steps

**Dataset**: Airline Passengers

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load and scale data
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url, parse_dates=["Month"], index_col="Month")
df.columns = ["passengers"]

scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df[["passengers"]]).flatten()

# Train/test split
train_size = len(data_scaled) - 24
train_data = data_scaled[:train_size]
test_data = data_scaled[train_size:]

## 1. Windowed Dataset

Convert time series into (input_window, target) pairs for supervised learning.

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, data, window_size=12):
        self.data = torch.FloatTensor(data)
        self.window_size = window_size
    
    def __len__(self):
        return len(self.data) - self.window_size
    
    def __getitem__(self, idx):
        x = self.data[idx:idx + self.window_size].unsqueeze(-1)  # (window, 1)
        y = self.data[idx + self.window_size]
        return x, y

window_size = 12
train_dataset = TimeSeriesDataset(train_data, window_size)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

print(f"Window size: {window_size}")
print(f"Training sequences: {len(train_dataset)}")
x_sample, y_sample = train_dataset[0]
print(f"Input shape: {x_sample.shape}, Target shape: {y_sample.shape}")

## 2. LSTM Forecaster

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout
        )
        self.fc = nn.Linear(hidden_dim, 1)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1, :]  # Take last time step
        return self.fc(last_hidden).squeeze(-1)

model = LSTMForecaster().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

losses = []
for epoch in range(100):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    scheduler.step(avg_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.6f}")

In [ ]:
# Recursive prediction on test set
model.eval()
predictions = []

# Start with last window of training data
input_seq = torch.FloatTensor(train_data[-window_size:]).unsqueeze(0).unsqueeze(-1).to(device)

with torch.no_grad():
    for _ in range(len(test_data)):
        pred = model(input_seq)
        predictions.append(pred.item())
        # Shift window: drop first, append prediction
        new_val = pred.view(1, 1, 1)
        input_seq = torch.cat([input_seq[:, 1:, :], new_val], dim=1)

# Inverse transform
predictions_orig = scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()
actual_orig = scaler.inverse_transform(test_data.reshape(-1, 1)).flatten()

# Plot
train_orig = df["passengers"][:train_size]

plt.figure(figsize=(12, 5))
plt.plot(df.index[:train_size], train_orig, label="Training", color="teal")
plt.plot(df.index[train_size:], actual_orig, label="Actual", color="black", linewidth=2)
plt.plot(df.index[train_size:], predictions_orig, label="LSTM Forecast", color="coral", linestyle="--")
plt.title("LSTM Time Series Forecast")
plt.legend()
plt.tight_layout()
plt.show()

mae = mean_absolute_error(actual_orig, predictions_orig)
rmse = np.sqrt(mean_squared_error(actual_orig, predictions_orig))
print(f"MAE: {mae:.1f}, RMSE: {rmse:.1f}")

## Key Takeaways

1. **Scale your data** - LSTMs work best with normalized inputs (MinMaxScaler or StandardScaler)
2. **Window size matters** - it determines how much history the model sees (often set to seasonal period)
3. **Recursive prediction accumulates error** - each prediction feeds into the next
4. **LSTMs shine with complex, nonlinear patterns** but ARIMA/Prophet are often better for simple seasonal data
5. **For production**: consider PyTorch Forecasting or Darts libraries for state-of-the-art temporal models